In [2]:
import sys
sys.path.insert(1, "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/tod")

import tod.corpus
import tod.outliers
import tod.clustering
import tod.plotting

corpus = tod.corpus.Corpus(
    treebank_path="/Users/madalina/Documents/M1TAL/stage-SK/Treebanks/UD_French-GSD-master",
    grew_pattern="pattern {X[upos=ADV]} without {X[InIdiom=Yes];X[Idiom=Yes]}",
    patterns_text_file="../3. probability_matrix/patterns_adv.txt",
    matrix_type="coverage"
)

X = corpus.feature_matrix

Number of matches after filtering: 11701


In [3]:
X.shape

(129, 298)

In [8]:
import tod.dimension_reduction_classic

dim_red = tod.dimension_reduction_classic.Tsne_corpus(corpus, n_components=2)
clustering = tod.clustering.KMeans(corpus, k=2)
fig = tod.plotting.cluster_scatter_plot(corpus, dim_red, clustering)

In [9]:
fig

In [10]:
fig.write_html("adverbs_kmeans.html")

In [11]:
import pysparcl
k = 2
tidy_k_perm = pysparcl.cluster.permute(X, k=k, nperms=25, nvals=10)

print("Best wbound:", tidy_k_perm['bestw'])

tidy_k_result = pysparcl.cluster.kmeans(X, k=k, wbounds=tidy_k_perm['bestw'])[0]
tidy_k_weights = tidy_k_result['ws']

tidy_k_clusters = {i: [] for i in range(k)}
for i in range(len(tidy_k_result['cs'])):
    cluster_id = tidy_k_result['cs'][i]
    tidy_k_clusters[cluster_id].append(corpus.idx2lexunit(i))

from sklearn.manifold import TSNE
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X)
# Create a DataFrame for Plotly
df = pd.DataFrame({
    'PCA1': X_tsne[:, 0],
    'PCA2': X_tsne[:, 1],
    'Cluster': tidy_k_result['cs'],
    'Word': [corpus.idx2lexunit(i) for i in range(len(tidy_k_result['cs']))]
})
fig = go.Figure()
for cluster in range(k):
    cluster_data = df[df['Cluster'] == cluster]
    fig.add_trace(go.Scatter(
        x=cluster_data['PCA1'],
        y=cluster_data['PCA2'],
        mode='markers',
        marker=dict(size=10),
        name=f'Cluster {cluster}',
        text=cluster_data['Word'],
        hovertemplate='%{text}<extra></extra>',
    ))

fig.update_layout(
    title='Word Clusters',
    xaxis_title='tsne1',
    yaxis_title='tsne2',
    # width=1600,  # Set the width of the figure
    # height=800,  # Set the height of the figure

    updatemenus=[
        {
            'buttons': [
                {
                    'label': 'All Clusters',
                    'method': 'update',
                    'args': [{'visible': [True] * k},
                             {'title': 'All Clusters'}]
                }
            ] + [
                {
                    'label': f'Cluster {i}',
                    'method': 'update',
                    'args': [{'visible': [j == i - 1 for j in range(k)]},
                             {'title': f'Cluster {i}'}]
                } for i in range(1, k + 1)
            ],
            'direction': 'down',
            'showactive': True
        }
    ]
)

fig.show()

Best wbound: 3.745232256134109


In [12]:
import numpy as np
tidy_k_important_features = np.argsort(-tidy_k_weights) 
for i in tidy_k_important_features[:10]:  
    print(f"{corpus.idx2feature(i)}: weight {tidy_k_weights[i]:.3f}")

node:X:child:rel_shallow=obl:arg: weight 0.554
node:X:child:upos=NOUN: weight 0.464
node:X:next:upos=ADP: weight 0.438
node:X:own:rel_shallow=advmod: weight 0.275
node:X:child:Number=Sing: weight 0.214
node:X:parent:position=after: weight 0.209
node:X:child:Gender=Masc: weight 0.198
node:X:next:Number=Sing: weight 0.135
node:X:parent:position=before: weight 0.124
node:X:child:Number=Plur: weight 0.095


In [13]:
cluster_assignments = {}
for orig_c, l in tidy_k_clusters.items():
    for lexunit in l:
        cluster_assignments[lexunit] = orig_c

In [14]:
import csv

  
with open("adverbs_cluster_assignments.csv", "w") as f:

	writer = csv.writer(f)

	for lexunit, cluster in cluster_assignments.items():

		writer.writerow([lexunit, cluster])